# Raviart–Thomas Darcy: RT0, RT1 and RT2

The MHM flux is locally RTm and its pressure is discontinuous Pm. The classical comparison is globally conforming RTm/Pm on the same fine spacing. Physical flux, pressure and divergence norms below use the exact sinusoidal solution.


The current main equations and their physical checks execute from source. The full acquisition and its field galleries run with `--study`; inspect `--study --plan` for locked commands and resource requirements. Exact original external comparisons require `--historical` and their attributed payloads.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/b16db72c4e51a643e179b04f45a1461b6348f258042148791ca2b90990b4e081/33_darcy_rt-companion.zip"
COMPANION_SHA256 = "b16db72c4e51a643e179b04f45a1461b6348f258042148791ca2b90990b4e081"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/33_darcy_rt.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
from IPython.display import Image, display
from pymhm.meshes.triangle import TriangleMesh


## Anisotropic affine patch

This check exercises physical flux orientation and exact RT representation; it is separate from the convergence study.


## Declare local and global forms

The application supplies the local mixed saddle for coefficients u=(q,p,r),
where r is the private boundary-pressure multiplier:

$$
\begin{aligned}
A_T&=\begin{bmatrix}M&-D^{\mathsf T}&N\\-D&0&0\\N^{\mathsf T}&0&0\end{bmatrix},
&f_T&=\begin{bmatrix}0\\-F\\0\end{bmatrix},\\
B_T&=\begin{bmatrix}0\\0\\-R\end{bmatrix},
&C_T&=-B_T^{\mathsf T}.
\end{aligned}
$$

Here M tests K inverse, D tests divergence, N selects exterior normal moments,
and R maps the fixed macroface flux orientation to outward fine-edge moments.
RT/BDM kernels supply their executed normal/interior basis; the pressure and
constant boundary-pressure modes form the declared joint kernel. Physical
pressure integrals provide its moments without volume normalization. The
global `Equation` supplies pressure boundary data; pure Neumann data add the
physical mean constraint. `assemble` executes these declarations and recovery
interprets the same H(div) flux and DG pressure coefficients.


In [ ]:
from pymhm import assemble
from examples.formulations.darcy import pressure_constraints
from examples.formulations.mixed_darcy import define_rt_darcy, recover_rt_darcy

K = np.array([[2.0, 0.3], [0.3, 1.0]])


def pressure(x):
    """Return an affine pressure with nonhomogeneous boundary data."""
    return 1 + x[:, 0] + 2 * x[:, 1]


for degree in (0, 1, 2):
    solution_definition = define_rt_darcy(
        TriangleMesh.unit_square(1), degree=degree, permeability=K, dirichlet=pressure
    )
    solution_system = assemble(solution_definition.problem)
    solution = recover_rt_darcy(
        solution_definition,
        solution_system,
        solution_system.solve(
            constraints=pressure_constraints(solution_definition, solution_system)
        ),
    )
    assert solution.flux_l2_error(-K @ np.array([1.0, 2.0])) < 1e-10
    assert max((np.max(np.abs(r)) for r in solution.fine_equilibrium_residuals())) < 1e-10
print("All three RT flux patches passed")


## Five levels against the analytical solution

The archived campaign has local refinement two and one degree-m polynomial per macroface. The classical method has no macroface restriction. Its own error is measured against the exact field; it is not treated as exact.


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    record = json.loads((root / "examples/results/darcy-rt.json").read_text())
    assert not record["source_changed_during_run"]
    assert len(record["rows"]) == 15
    for degree in (0, 1, 2):
        rows = [row for row in record["rows"] if row["degree"] == degree]
        assert [row["n"] for row in rows] == [1, 2, 4, 8, 16]
        for method in ("mhm", "classical"):
            assert rows[-1][method]["flux_l2_error"] < rows[0][method]["flux_l2_error"]
            assert max(row[method]["fine_moment_residual"] for row in rows) < 1e-10
        print(degree, rows[-1]["mhm"], rows[-1]["classical"])
    last = record["rows"][-1]
    archive = root / "examples/results" / last["fields"]
    assert hashlib.sha256(archive.read_bytes()).hexdigest() == last["fields_sha256"]


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    display(Image(filename=str(root / "docs/figures/darcy-rt/convergence.png")))
    display(Image(filename=str(root / "docs/figures/darcy-rt/fields.png")))


The field panels show individual fine-cell centroid samples and actual macro edges. Volume norms use complete RT/Pm polynomials, preserving discontinuous pressure and one-sided material values. Native optional tests independently assemble RT0–RT2 with DOLFINx/UFL.
